This notebook enables to generate scenario_specification.csv

In [6]:
import os
import pandas as pd
from itertools import product

In [7]:
folder = 'guinea'
file = 'scenarios.csv'

file = os.path.join(folder, file)
data = pd.read_csv(file)

In [ ]:
def combine_scenarios_with_inputs(df):
    """Use to build a scenario specification file from a scenarios.csv file.
    

    Combines all combinations of scenarios and outputs a dataframe in the same format
    as the input, but with additional columns for each scenario combination.

    Parameters:
    - df (DataFrame): A dataframe with `paramNames` as rows and scenarios as columns.

    Returns:
    - output_df (DataFrame): A dataframe with combined scenario columns (S1, S2, ...).
    """
    # Identify scenario columns (excluding the 'paramNames' column)
    scenario_columns = df.columns[1:]  # Assuming first column is 'paramNames'
    
    # Create all combinations of scenario activations (e.g., [0, 1, 0, 1])
    scenario_combinations = list(product([0, 1], repeat=len(scenario_columns)))
    
    # Initialize the output dataframe
    output_df = df.copy()
    
    # Add new scenario columns (e.g., "S1", "S2", ...)
    for i, combination in enumerate(scenario_combinations):
        scenario_name = f"S{i+1}"
        new_column = []
        
        # For each parameter, combine inputs for active scenarios
        for _, row in df.iterrows():
            relevant_files = []  # Collect all inputs for the parameter
            
            for col, is_active in zip(scenario_columns, combination):
                if is_active and pd.notna(row[col]):
                    relevant_files.append(row[col])
            
            # Join relevant files into a single string or keep None if no files
            if relevant_files:
                new_column.append(";".join(relevant_files))
            else:
                new_column.append(None)
        
        # Add the new column to the output dataframe
        output_df[scenario_name] = new_column
    
    return output_df

In [21]:
specification = combine_scenarios_with_inputs(data)
specification.dropna(axis=1, how='all', inplace=True)
specification.drop([i for i in data.columns if i != 'paramNames'], axis=1, inplace=True)
specification.to_csv(os.path.join(folder, 'scenarios_specification_{}.csv'.format(folder)), index=False)